## Exploring predictions

In [ ]:
import os
import sys
import gc
import random
import importlib as imp
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import silence_tensorflow

import experiment_settings
import build_model
import build_data
import plots
import methods
import save_files

from sklearn.metrics import mean_squared_error, mean_absolute_error

In [ ]:
print(f"python version = {sys.version}")
print(f"numpy version = {np.__version__}")
print(f"tensorflow version = {tf.__version__}")  

# tf.config.set_visible_devices([], "GPU")  # turn-off tensorflow-metal if it is on
print(tf.config.list_physical_devices('GPU'))

In [ ]:
# GET SETTINGS
EXP_NAME = "exp0"
settings = experiment_settings.get_settings(EXP_NAME)

TILE_LEN_DEG = 1.

directory_paths = methods.get_directories()
SAVE_MODEL_DIRECTORY = directory_paths["save_model_dir"]
DATA_DIRECTORY = directory_paths["data_dir"]
FIGURE_DIRECTORY = directory_paths["figures_dir"]
PREDICTIONS_DIRECTORY = directory_paths["predictions_dir"]

In [ ]:
# GET THE DATA
import read_landsat
imp.reload(build_data)
imp.reload(read_landsat)

for year in (2020,):#np.arange(2000,2023):

    print(' --- ' + str(year) + '---')
    settings["testing_years"] = (year, )
    settings["batch_size"] = 32
    settings["mode"] = "inference"
    settings["latlon_bounds"] = (-6.999, -6.0, 106.00, 106.999)
    
    # SET RANDOM SEEDS
    np.random.seed(settings["rng_seed"])
    random.seed(settings["rng_seed"])
    tf.random.set_seed(settings["rng_seed"])

    min_latfile = np.floor(settings["latlon_bounds"][0] / TILE_LEN_DEG) * TILE_LEN_DEG
    max_latfile = np.ceil(settings["latlon_bounds"][1] / TILE_LEN_DEG) * TILE_LEN_DEG
    min_lonfile = np.floor(settings["latlon_bounds"][2] / TILE_LEN_DEG) * TILE_LEN_DEG
    max_lonfile = np.ceil(settings["latlon_bounds"][3] / TILE_LEN_DEG) * TILE_LEN_DEG

    # FIXME: currently not suing this loop
    for latfile in np.arange(min_latfile + TILE_LEN_DEG, max_latfile + TILE_LEN_DEG, TILE_LEN_DEG):
        for lonfile in np.arange(min_lonfile, max_lonfile, TILE_LEN_DEG):

            # FIXME: this is overriding everyting and needs to be written to work in batch sizes
            # that equally fit
            settings["latlon_bounds"] = (-6.95, -6.05, 106.05, 106.95)

            # GET THE SAMPLE TAGS
            tags_test, __ = build_data.get_tags(settings)
            tfds_test = build_data.build_tf_dataset(settings, tags_test, settings["batch_size"])
            tfds_test = tfds_test.prefetch(tf.data.AUTOTUNE)

            # LOAD THE MODEL AND MAKE PREDICTIONS
            tf.keras.backend.clear_session()
            model = build_model.build_model(settings, 
                                            input_shape=np.shape(next(tfds_test.as_numpy_iterator())[0])[1:])

            checkpoint_dir = SAVE_MODEL_DIRECTORY + settings["exp_name"] + '/'
            model.load_weights(tf.train.latest_checkpoint(checkpoint_dir)).expect_partial()

            hfi_predict = model.predict(tfds_test, verbose=1)[:,0]
            __ = gc.collect()

            # SAVE THE PREDICTIONS AS A TIF
            predictions_filename = "predictions_" + tags_test[-1][0]
            hfi_predict, hfi_labels, lat0, lat1, lon0, lon1 = save_files.save_predictions_tif(settings, hfi_predict, predictions_filename)

            # PLOT THE RESULTS
            plt.figure(figsize=(10,5))
            plt.subplot(1,2,1)
            plots.plot_hfi_tile(hfi_predict, [lon0, lon1, lat1, lat0])
            plt.title('mlHFI Predictions for ' + str(settings["testing_years"][0]))
            plt.clim(0,100)

            plt.subplot(1,2,2)
            plots.plot_hfi_tile(hfi_labels, [lon0, lon1, lat1, lat0])
            plt.title('HFI Labels for ' + str(settings["testing_years"][0]))
            plt.clim(0,100)

            plt.savefig(FIGURE_DIRECTORY + predictions_filename + ".png")
            plt.show()
            # plt.close()